# 🏎️ JetBot 期末專案多功能控制面板 (分測 + 合併最終版)

本筆記本將整合程式分為三個部分，供您在實車落地時進行分步測試與最終合併演示：
- **單元一：純道路跟隨測試 (測試 PID 與轉向)** — 僅載入 ResNet 模型，不佔用 YOLO 與 PyCUDA 資源，適合調校 P/D Gain。
- **單元二：純交通路牌辨識測試 (測試 YOLO 與動作)** — 僅載入 YOLO 模型，測試路標停/走動作是否正常。
- **單元三：合併最終版 (道路跟隨 + 路牌辨識)** — 雙模型並行推論，進行期末考完整賽道演示。

> ⚠️ **重要守則**：每當您要從一個單元切換到另一個單元時，**必須執行該單元最下方的「安全停機與釋放相機」單元格**，否則會出現相機被佔用 (Resource busy) 的錯誤！

---

## 🛠️ 零、道路跟隨 TensorRT 模型優化轉換 (選用)

如果您剛在 PC 端訓練了新的 `.pth` 模型，並將其上傳到了 `road_following_model/best_steering_model_xy.pth`，您**必須**在車端（JetBot）上執行以下儲存格，將其編譯優化為 `best_steering_model_xy_trt.pth`，才能運行後續的單元。

*註：此轉換步驟只能在 JetBot (Jetson Nano) 上執行，因為它需要實體 GPU 與 TensorRT 環境。在 PC 端直接執行會報 `ModuleNotFoundError: No module named 'torch2trt'` 錯誤。*

In [ ]:
# 0-1. 執行 TensorRT 編譯與轉換
import os
import torch
import torchvision.models as models
try:
    from torch2trt import torch2trt
    has_torch2trt = True
except ImportError:
    has_torch2trt = False

PYTORCH_MODEL = 'road_following_model/best_steering_model_xy.pth'
TRT_MODEL = 'road_following_model/best_steering_model_xy_trt.pth'

if not has_torch2trt:
    print("❌ 錯誤：找不到 torch2trt 模組！")
    print("請確認此儲存格是否是在 JetBot 自走車上執行。PC 端不支援此轉換。")
elif not os.path.exists(PYTORCH_MODEL):
    print(f"❌ 錯誤：找不到 PyTorch 原始權重檔案：{PYTORCH_MODEL}")
    print("請確認您已透過 WinSCP 或 FileZilla 將訓練好的 .pth 檔案上傳至對應目錄。")
else:
    print(f"正在載入 PyTorch 權重：{PYTORCH_MODEL}...")
    # 建立 ResNet-18 架構
    model = models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, 2)
    model.load_state_dict(torch.load(PYTORCH_MODEL))
    
    # 搬移至 GPU 並設定為評估模式
    model = model.cuda().eval().half()
    
    # 建立 TensorRT 編譯需要的範例輸入 (batch=1, 3 channels, 224x224)
    sample_data = torch.zeros((1, 3, 224, 224)).cuda().half()
    
    print("正在將模型編譯為 TensorRT 引擎 (這在 JetBot 上可能需要 1~3 分鐘，請耐心等候)...")
    model_trt = torch2trt(model, [sample_data], fp16_mode=True)
    
    # 儲存優化後的模型
    os.makedirs(os.path.dirname(TRT_MODEL), exist_ok=True)
    torch.save(model_trt.state_dict(), TRT_MODEL)
    print(f"✓ 轉換成功！已儲存 TensorRT 優化模型至：{TRT_MODEL}")

---

## 🏁 單元一：純道路跟隨測試 (調校 PID 參數)
此部分只載入 Project 5 的道路循跡模型，適合微調馬達轉向與 PD 參數。

In [ ]:
# A1. 匯入基礎套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

# A2. 載入 ResNet-18 TensorRT 模型
device = torch.device('cuda')
model_road = TRTModule()
model_road.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))
print("單元一：道路跟隨模型載入成功 ✓")

In [ ]:
# A3. 初始化相機 (224x224) 與 Robot
camera_r = Camera.instance(width=224, height=224, fps=10)
robot_r = Robot()
print("相機 (224x224) 與馬達初始化完成 ✓")

In [ ]:
# A4. 定義影像處理與滑桿
mean_r = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_r = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road_only(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).float().to(device).half()
    image = image / 255.0
    image = (image - mean_r[:, None, None]) / std_r[:, None, None]
    return image.unsqueeze(0)

speed_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Speed Gain')
p_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.08, description='P Gain')
d_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.005, value=0.00, description='D Gain')
bias_slider_r = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Motor Bias')
image_widget_r = widgets.Image(format='jpeg', width=224, height=224)

display(image_widget_r, speed_slider_r, p_slider_r, d_slider_r, bias_slider_r)

In [ ]:
# A5. 循跡控制迴圈
angle_last_r = 0.0

def execute_road_only(change):
    global angle_last_r
    image = change['new']
    
    road_tensor = preprocess_road_only(image)
    output = model_road(road_tensor).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[1])
    
    y_adjusted = (0.5 - y) / 2.0
    angle = np.arctan2(x, y_adjusted)
    
    pid = angle * p_slider_r.value + (angle - angle_last_r) * d_slider_r.value
    angle_last_r = angle
    steering = pid + bias_slider_r.value
    
    base_speed = speed_slider_r.value
    left_motor = max(min(base_speed + steering, 1.0), 0.0)
    right_motor = max(min(base_speed - steering, 1.0), 0.0)
    
    robot_r.left_motor.value = left_motor
    robot_r.right_motor.value = right_motor
    
    # 繪製綠色路徑點
    display_img = np.copy(image)
    pixel_x = int(x * 112 + 112)
    pixel_y = int(y_adjusted * 112 + 112)
    cv2.circle(display_img, (pixel_x, pixel_y), 8, (0, 255, 0), -1)
    image_widget_r.value = bgr8_to_jpeg(display_img)

print("單元一控制函式 execute_road_only 定義完成 ✓")

In [ ]:
# A6. 啟動單元一 (道路跟隨)
execute_road_only({'new': camera_r.value})
camera_r.observe(execute_road_only, names='value')
print("🚗 單元一：純道路跟隨自動駕駛啟動！")

In [ ]:
# A7. 安全停止單元一 (切換單元前必執行！)
camera_r.unobserve(execute_road_only, names='value')
time.sleep(0.5)
robot_r.stop()
camera_r.stop()
print("單元一已停止，相機資源已釋放 ✓")

---

## 🚥 單元二：純交通路牌辨識測試
此部分只載入 Project 6 的 YOLO 號誌偵測模型，測試自走車看到路牌後的停、走動作是否完全正確。在此模式下，自走車會直線行駛，只反應路牌動作。

In [ ]:
# B1. 匯入 YOLO 相關套件
import os
import time
import cv2
import numpy as np
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot
import pycuda.autoinit
from utils.yolo import TRT_YOLO

# B2. 載入 YOLO 模型
model_sign_only = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
print("單元二：路牌辨識模型載入成功 ✓")

In [ ]:
# B3. 初始化相機 (416x416) 與 Robot
camera_s = Camera.instance(width=416, height=416, fps=10)
robot_s = Robot()
print("相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# B4. 建立滑桿介面
speed_slider_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Base Speed')
alert_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')
image_widget_s = widgets.Image(format='jpeg', width=416, height=416)

display(image_widget_s, speed_slider_s, alert_width_slider_s)

In [ ]:
# B5. 路牌偵測控制迴圈
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0

def execute_sign_only(change):
    global stop_cooldown_s, rail_cooldown_s
    img = change['new']
    current_time = time.time()
    
    # 偵測路牌
    boxes, confs, clss = model_sign_only.detect(img, conf_th=0.6)
    
    detected_signs = []
    for box, conf, cls in zip(boxes, confs, clss):
        width = box[2] - box[0]
        detected_signs.append({"width": width, "class_id": cls, "box": box})
    detected_signs.sort(reverse=True, key=lambda x: x["width"])
    
    speed_multiplier = 1.0
    current_action = "Driving Straight"
    
    active_sign = None
    ALERT_WIDTH = alert_width_slider_s.value
    
    for sign in detected_signs:
        cid = sign["class_id"]
        w = sign["width"]
        
        if cid == 3 and current_time < stop_cooldown_s: continue
        if cid == 2 and current_time < rail_cooldown_s: continue
        
        if w >= ALERT_WIDTH:
            active_sign = sign
            break
            
    if active_sign is not None:
        cid = active_sign["class_id"]
        
        if cid == 3:  # stop (停車再開) ➡ 原地停止 2 秒
            current_action = "ACTION: Stop (2s)"
            print("[SIGN] Detected STOP. Stopping for 2s...")
            robot_s.stop()
            time.sleep(2.0)
            stop_cooldown_s = time.time() + 10.0
        elif cid == 2:  # rail (鐵路平交道) ➡ 原地停止 5 秒
            current_action = "ACTION: Railway (5s)"
            print("[SIGN] Detected RAILWAY. Stopping for 5s...")
            robot_s.stop()
            time.sleep(5.0)
            rail_cooldown_s = time.time() + 10.0
        elif cid == 1:  # pedestrian (當心行人) ➡ 減速 0.7 倍
            current_action = "ACTION: Pedestrian (Slow 0.7x)"
            speed_multiplier = 0.7
        elif cid == 0:  # blocked (道路封閉) ➡ 永久停止
            current_action = "ACTION: Blocked! STOP"
            print("[SIGN] Detected BLOCKED. Stopping permanently.")
            robot_s.stop()
            camera_s.unobserve(execute_sign_only, names='value')
            return
            
    # 驅動馬達直行 (不修正方向)
    base_speed = speed_slider_s.value * speed_multiplier
    robot_s.left_motor.value = base_speed
    robot_s.right_motor.value = base_speed
    
    # 標記繪圖
    display_img = np.copy(img)
    for sign in detected_signs:
        box = sign["box"]
        cid = sign["class_id"]
        cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
        cv2.putText(display_img, f"ID:{cid} W:{sign['width']}", (box[0], box[1] - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    image_widget_s.value = bgr8_to_jpeg(display_img)

print("單元二控制函式 execute_sign_only 定義完成 ✓")

In [ ]:
# B6. 啟動單元二 (路牌辨識)
execute_sign_only({'new': camera_s.value})
camera_s.observe(execute_sign_only, names='value')
print("🚗 單元二：純路牌辨識偵測啟動！")

In [ ]:
# B7. 安全停止單元二 (切換單元前必執行！)
camera_s.unobserve(execute_sign_only, names='value')
time.sleep(0.5)
robot_s.stop()
camera_s.stop()
print("單元二已停止，相機資源已釋放 ✓")

---

## 🏁🏁 單元三：合併最終版 (道路跟隨 + 路牌辨識)
雙模型載入並行推論。用大圖 (416x416) 偵測路牌，偵測後即時下採樣 (224x224) 預測方向，配合 PD 運算驅動馬達。此部分為期末驗收使用的最終演示版本。

In [ ]:
# C1. 匯入完整套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot
import pycuda.autoinit
from utils.yolo import TRT_YOLO

# C2. 同時載入兩個 TensorRT 模型
device = torch.device('cuda')
model_road_final = TRTModule()
model_road_final.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))
print("道路模型載入完成 ✓")

model_sign_final = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
print("路牌模型載入完成 ✓")

In [ ]:
# C3. 初始化相機 (416x416) 與 Robot
camera_f = Camera.instance(width=416, height=416, fps=10)
robot_f = Robot()
print("最終合併版相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# C4. 正規化與滑桿介面
mean_f = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_f = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road_final(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).float().to(device).half()
    image = image / 255.0
    image = (image - mean_f[:, None, None]) / std_f[:, None, None]
    return image.unsqueeze(0)

speed_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Speed Gain')
p_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.08, description='P Gain')
d_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.005, value=0.00, description='D Gain')
bias_slider_f = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Motor Bias')
alert_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')
image_widget_f = widgets.Image(format='jpeg', width=416, height=416)

display(
    image_widget_f, 
    speed_slider_f, 
    p_slider_f, 
    d_slider_f, 
    bias_slider_f, 
    alert_width_slider_f
)

In [ ]:
# C5. 合併自動駕駛核心迴圈
angle_last_f = 0.0
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0

def execute_final(change):
    global angle_last_f, stop_cooldown_f, rail_cooldown_f
    img = change['new']
    current_time = time.time()
    
    # 1. 偵測路牌
    boxes, confs, clss = model_sign_final.detect(img, conf_th=0.6)
    
    detected_signs = []
    for box, conf, cls in zip(boxes, confs, clss):
        width = box[2] - box[0]
        detected_signs.append({"width": width, "class_id": cls, "box": box})
    detected_signs.sort(reverse=True, key=lambda x: x["width"])
    
    speed_multiplier = 1.0
    current_action = "Following Lane"
    
    active_sign = None
    ALERT_WIDTH = alert_width_slider_f.value
    
    for sign in detected_signs:
        cid = sign["class_id"]
        w = sign["width"]
        
        if cid == 3 and current_time < stop_cooldown_f: continue
        if cid == 2 and current_time < rail_cooldown_f: continue
        
        if w >= ALERT_WIDTH:
            active_sign = sign
            break
            
    if active_sign is not None:
        cid = active_sign["class_id"]
        
        if cid == 3:  # stop (停車再開) ➡ 原地停止 2 秒
            current_action = "ACTION: Stop (2s)"
            print("[SIGN] Detected STOP. Stopping for 2s...")
            robot_f.stop()
            time.sleep(2.0)
            stop_cooldown_f = time.time() + 10.0
            
        elif cid == 2:  # rail (鐵路平交道) ➡ 原地停止 5 秒
            current_action = "ACTION: Railway (5s)"
            print("[SIGN] Detected RAILWAY. Stopping for 5s...")
            robot_f.stop()
            time.sleep(5.0)
            rail_cooldown_f = time.time() + 10.0
            
        elif cid == 1:  # pedestrian (當心行人) ➡ 減速 0.7 倍
            current_action = "ACTION: Pedestrian (Slow 0.7x)"
            speed_multiplier = 0.7
            
        elif cid == 0:  # blocked (道路封閉) ➡ 永久停止
            current_action = "ACTION: Blocked! STOP"
            print("[SIGN] Detected BLOCKED. Stopping permanently.")
            robot_f.stop()
            camera_f.unobserve(execute_final, names='value')
            return
            
    # 2. 道路循跡推論
    road_tensor = preprocess_road_final(img)
    output = model_road_final(road_tensor).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[1])
    
    y_adjusted = (0.5 - y) / 2.0
    angle = np.arctan2(x, y_adjusted)
    
    pid = angle * p_slider_f.value + (angle - angle_last_f) * d_slider_f.value
    angle_last_f = angle
    steering = pid + bias_slider_f.value
    
    base_speed = speed_slider_f.value * speed_multiplier
    left_motor = max(min(base_speed + steering, 1.0), 0.0)
    right_motor = max(min(base_speed - steering, 1.0), 0.0)
    
    robot_f.left_motor.value = left_motor
    robot_f.right_motor.value = right_motor
    
    # 3. 繪圖標記
    display_img = np.copy(img)
    # 標記路徑中心點 (綠圓點)
    pixel_x = int(x * 208 + 208)
    pixel_y = int(y_adjusted * 208 + 208)
    cv2.circle(display_img, (pixel_x, pixel_y), 10, (0, 255, 0), -1)
    
    # 標記號誌邊界框 (紅框)
    for sign in detected_signs:
        box = sign["box"]
        cid = sign["class_id"]
        cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
        cv2.putText(display_img, f"ID:{cid} W:{sign['width']}", (box[0], box[1] - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                    
    cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    image_widget_f.value = bgr8_to_jpeg(display_img)

print("單元三控制函式 execute_final 定義完成 ✓")

In [ ]:
# C6. 啟動單元三 (最終合併自動駕駛)
execute_final({'new': camera_f.value})
camera_f.observe(execute_final, names='value')
print("🚗 單元三：雙模型合併自動駕駛啟動！")

In [ ]:
# C7. 安全停止單元三
camera_f.unobserve(execute_final, names='value')
time.sleep(0.5)
robot_f.stop()
camera_f.stop()
print("單元三已停止，相機與馬達資源已安全釋放 ✓")